# VestiAI — IDM-VTON Virtual Try-On Pipeline on Google Colab

**Project:** VestiAI — AI-Powered Virtual Clothing Try-On for E-Commerce  
**Objective:** Run real 2D Virtual Try-On inference using **IDM-VTON** on a free Google Colab T4 GPU (16 GB VRAM).

---

## 1. Verify GPU Hardware
Make sure **Runtime > Change runtime type > Hardware accelerator** is set to **T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Clone IDM-VTON Repository & Install Dependencies

In [ ]:
# Clone official IDM-VTON repo
!git clone https://github.com/yisol/IDM-VTON.git
%cd IDM-VTON

# Install pinned dependencies
!pip install -q diffusers==0.27.2 transformers==4.38.2 accelerate==0.27.2 torchvision==0.17.2
!pip install -q einops bitsandbytes gradio pyngrok fastapi uvicorn pillow

## 3. Load IDM-VTON Pipeline with Diffusers & Hugging Face Checkpoint

In [ ]:
import torch
from PIL import Image
from diffusers import AutoencoderKL, DDPMScheduler
import os

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 4. Run Sample Upper-Body Try-On Inference

In [ ]:
# Run official inference on provided test pair (person & garment)
!python inference.py \
    --pretrained_model_name_or_path "yisol/IDM-VTON" \
    --width 768 \
    --height 1024 \
    --num_inference_steps 30 \
    --guidance_scale 2.0 \
    --seed 42

## 5. Inspect and Display Try-On Result Comparison

In [ ]:
import matplotlib.pyplot as plt
import glob

# Find generated image
output_images = glob.glob("result/*.png") + glob.glob("result/*.jpg")
if output_images:
    result_img = Image.open(output_images[0])
    plt.figure(figsize=(10, 12))
    plt.imshow(result_img)
    plt.axis('off')
    plt.title("VestiAI Real IDM-VTON Try-On Result (Google Colab T4 GPU)")
    plt.show()
else:
    print("Run inference step to generate output.")

## 6. (Optional) Expose Colab as a Remote GPU Inference Worker for VestiAI API
This runs a lightweight FastAPI microservice with pyngrok, allowing your local VestiAI backend to send try-on jobs directly to this Colab GPU!

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
import uvicorn
import threading
from pyngrok import ngrok

app = FastAPI(title="VestiAI Colab Worker")

@app.get("/health")
def health():
    return {"status": "ready", "gpu": torch.cuda.get_device_name(0)}

# Set ngrok token if available, then run on port 8000
# public_url = ngrok.connect(8000)
# print(f"Remote Colab Inference URL: {public_url}")